In [ ]:
import sys
from pathlib import Path

# Ruta relativa desde notebooks/ hasta la raíz del proyecto
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports limpios gracias al __init__.py
from src import ModeloSociedad, Trabajador, Empresario, Antisistema

from mesa.visualization import SolaraViz, make_space_component, make_plot_component
from IPython.display import display, HTML
import numpy as np

# Imports de métricas
#from src.metrics import calcular_gini, grievance_medio, detectar_revueltas_activas

In [ ]:
# Parámetros de configuración de la simulación
#Tamaño del tablero en el que pueden estar los agentes
anchura = 20
altura = 20

#Semilla de la pseudoaleatoriedad para poder repetir resultados en las ejecuciones
semilla = 150

#Cantidad de acciones que realizará cada agente (se realiza una acción cada 1.0 time steps)
time_steps = 100


# Parámetros de configuración de los agentes en general
#Variables relacionadas con el tiempo del que dispone el agente para actuar cada día
tiempoMaxPosible = 24                                        #Tiempo en horas que el agente tiene disponibles en un dia
tiempoVital = 8                                              #Tiempo en horas que se utiliza en hacer acciones necesarias para la supervivencia (comida, higiene, etc.)

#Energia para realizar las acciones
energia = 100

#Cantidad de cada tipo de agente
n_trabajadores = 100                  #Cantidad de trabajadores
n_empresarios = 20                    #Cantidad de empresarios
n_rebeldes = 10                       #Cantidad de rebeldes

porcentajeAleatorio = 0.5

# Parámetros de configuración de agentes específicos
#Trabajador
dineroInicialTrabajador = 500
insatisfaccionInicialTrabajador = 15

tiempoTrabajo = 8           #Horas que dedica a trabajar de manera directa
maxTiempoAlTrabajo = 1.5    #Cantidad máxima de tiempo en horas que puede tardar el agente en transportarse al trabajo (ida + vuelta)


#Empresario
dineroInicialEmpresario = 15000
insatisfaccionInicialEmpresario = 0


#Antisistema
dineroInicialAntisistema = 50
insatisfaccionInicialAntisistema = 50

In [ ]:
#Instanciamos la simulación
modeloSociedad = ModeloSociedad(n_trabajadores=n_trabajadores, n_empresarios=n_empresarios, n_rebeldes=n_rebeldes, anchuraGrid=anchura, alturaGrid=altura, seed=semilla,
                                tiempoMaxPosible=tiempoMaxPosible, tiempoVital=tiempoVital, energia=energia, porcentajeAleatorio=porcentajeAleatorio,
                                dineroInicialT=dineroInicialTrabajador, insatisfaccionInicialT=insatisfaccionInicialTrabajador, tiempoTrabajo=tiempoTrabajo, maxTiempoAlTrabajo=maxTiempoAlTrabajo,
                                dineroInicialE=dineroInicialEmpresario, insatisfaccionInicialE=insatisfaccionInicialEmpresario,
                                dineroInicialA=dineroInicialAntisistema, insatisfaccionInicialA=insatisfaccionInicialAntisistema)

In [ ]:
display(HTML("""
<style>
    .solara-component, .vue-component, .jupyter-widgets, .solara-viz-container {
        width: 100% !important;
        min-height: 900px !important;
        height: 920px !important;
        max-height: none !important;
    }
    .solara-app {
        max-width: 100% !important;
        padding: 0 !important;
    }
    .jp-OutputArea-output {
        max-height: none !important;
        overflow: visible !important;
    }
</style>
"""))

def agent_portrayal(agent):
    if agent.tipo == "Trabajador":
        color = "#1f77b4"
        size = 7
    elif agent.tipo == "Empresario":
        color = "#2ca02c"
        size = 9 + min(15, getattr(agent, 'riqueza', 0) / 1200)
    elif agent.tipo == "Antisistema":
        color = "#d62728"
        size = 9
    else:
        color = "gray"
        size = 6
    
    alpha = 0.7 + (getattr(agent, 'insatisfaccion', 0) / 300)
    alpha = min(1.0, alpha)
    
    return {
        "color": color,
        "size": size,
        "alpha": alpha,
        "marker": "o",
        "zorder": 10 if agent.tipo == "Antisistema" else 5
    }


# Componente del espacio (el grid con los agentes)
# Este es el componente principal que muestra el mapa
space_component = make_space_component(agent_portrayal=agent_portrayal)

# Componente de gráfico (evolución temporal)
# Muestra el gráfico de "Insatisfaccion_Media" que definiste en el DataCollector
plot_insatisfaccion = make_plot_component("Insatisfaccion_Media")


# Parámetros ajustables desde la interfaz (sliders)
# Estos sliders permiten cambiar valores y volver a ejecutar la simulación
model_params = {
    "n_trabajadores": {
        "type": "SliderInt",
        "value": 100,
        "label": "Número de Trabajadores",
        "min": 50,
        "max": 400,
        "step": 10
    },
    "n_empresarios": {
        "type": "SliderInt",
        "value": 25,
        "label": "Número de Empresarios",
        "min": 10,
        "max": 120,
        "step": 5
    },
    "n_rebeldes": {
        "type": "SliderInt",
        "value": 15,
        "label": "Número de Antisistema/Rebeldes",
        "min": 5,
        "max": 100,
        "step": 5
    },
    "anchuraGrid": {
        "type": "SliderInt",
        "value": 30,
        "label": "Anchura del Grid",
        "min": 20,
        "max": 60,
        "step": 5
    },
    "alturaGrid": {
        "type": "SliderInt",
        "value": 30,
        "label": "Altura del Grid",
        "min": 20,
        "max": 60,
        "step": 5
    },
    "seed": {
        "type": "SliderInt",
        "value": 42,
        "label": "Semilla (Reproducibilidad)",
        "min": 1,
        "max": 9999,
        "step": 1
    }
}


# Crear la visualización interactiva
page = SolaraViz(
    model=modeloSociedad,
    model_params=model_params,
    measures=[space_component, plot_insatisfaccion],
    name="Simulación de Sociedad",
    play_interval=180,
    height=950,           # Altura muy grande
    width="100%"
)


# Mostrar la visualización interactiva en el notebook
page

In [ ]:
model_df = page.model.datacollector.get_model_vars_dataframe()
agent_df = page.model.datacollector.get_agent_vars_dataframe()

print("Datos recolectados:")
display(model_df.tail())
display(agent_df.head())